In [1]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Load data
# -----------------------------
print("Loading data...")

train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


# -----------------------------
# Prepare features
# -----------------------------
spectral_cols = [
    c for c in train.columns
    if c not in ["sample number", "species number", "樹種", "含水率"]
]

X = train[spectral_cols].values
y = train["含水率"].values
X_test = test[spectral_cols].values

print("Spectral features:", X.shape[1])


# -----------------------------
# Scale features
# -----------------------------
print("Scaling features...")

scaler = StandardScaler()

X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)


# -----------------------------
# Ensemble configuration
# -----------------------------
alphas = [0.001, 0.002, 0.004, 0.006, 0.01]
seeds = [42, 77, 123]

kf = KFold(n_splits=8, shuffle=True, random_state=42)

test_preds = np.zeros(len(X_test))
total_models = 0


# -----------------------------
# Training ensemble
# -----------------------------
print("\nRunning ElasticNet ensemble...")

for alpha in alphas:

    for seed in seeds:

        print(f"\nModel alpha={alpha} seed={seed}")

        fold_preds = np.zeros(len(X_test))
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model = ElasticNet(
                alpha=alpha,
                l1_ratio=0.95,
                max_iter=20000,
                tol=1e-3,
                selection="random",
                random_state=seed
            )

            model.fit(X_train, y_train)

            preds = model.predict(X_val)

            rmse = np.sqrt(mean_squared_error(y_val, preds))
            fold_scores.append(rmse)

            fold_preds += model.predict(X_test) / kf.n_splits

        print("CV RMSE:", np.mean(fold_scores))

        test_preds += fold_preds
        total_models += 1


# -----------------------------
# Average ensemble predictions
# -----------------------------
test_preds /= total_models


# -----------------------------
# Save submission
# -----------------------------
os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": test_preds
})

path = "../submissions/exp20_elasticnet_big_ensemble.csv"

submission.to_csv(path, index=False, header=False)

print("\nSaved submission:", path)
print(pd.read_csv(path, header=None).head())

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Spectral features: 1555
Scaling features...

Running ElasticNet ensemble...

Model alpha=0.001 seed=42
CV RMSE: 11.926496517829495

Model alpha=0.001 seed=77
CV RMSE: 11.928892640729151

Model alpha=0.001 seed=123
CV RMSE: 11.929999105577522

Model alpha=0.002 seed=42
CV RMSE: 12.606273302669724

Model alpha=0.002 seed=77
CV RMSE: 12.606460485823872

Model alpha=0.002 seed=123
CV RMSE: 12.60667185115069

Model alpha=0.004 seed=42
CV RMSE: 13.813339735001527

Model alpha=0.004 seed=77
CV RMSE: 13.814038950794243

Model alpha=0.004 seed=123
CV RMSE: 13.814146905644233

Model alpha=0.006 seed=42
CV RMSE: 14.502085171000875

Model alpha=0.006 seed=77
CV RMSE: 14.502658965465024

Model alpha=0.006 seed=123
CV RMSE: 14.504146247243213

Model alpha=0.01 seed=42
CV RMSE: 15.420940297242144

Model alpha=0.01 seed=77
CV RMSE: 15.423202943589377

Model alpha=0.01 seed=123
CV RMSE: 15.423773096914918

Saved submission: ../submission

In [3]:
import pandas as pd
import numpy as np
import os
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)


# -----------------------------
# Metric diagnostics function
# -----------------------------
def metric_diagnostics(y_true, y_pred):

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)

    # RMSLE
    y_true_log = np.log1p(y_true)
    y_pred_log = np.log1p(np.maximum(y_pred, 0))
    rmsle = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    # normalized RMSE
    nrmse_mean = rmse / np.mean(y_true)
    nrmse_range = rmse / (np.max(y_true) - np.min(y_true))

    print("\nMetric diagnostics")
    print("------------------")
    print("RMSE:", rmse)
    print("MAE:", mae)
    print("RMSLE:", rmsle)
    print("NRMSE (mean):", nrmse_mean)
    print("NRMSE (range):", nrmse_range)


# -----------------------------
# Load data
# -----------------------------
print("Loading data...")

train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)


# -----------------------------
# Prepare features
# -----------------------------
spectral_cols = [
    c for c in train.columns
    if c not in ["sample number", "species number", "樹種", "含水率"]
]

X = train[spectral_cols].values
y = train["含水率"].values
X_test = test[spectral_cols].values

print("Spectral features:", X.shape[1])


# -----------------------------
# Scale features
# -----------------------------
print("Scaling features...")

scaler = StandardScaler()

X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)


# -----------------------------
# Ensemble configuration
# -----------------------------
alphas = [0.001, 0.002, 0.004, 0.006, 0.01]
seeds = [42, 77, 123]

kf = KFold(n_splits=8, shuffle=True, random_state=42)

test_preds = np.zeros(len(X_test))
total_models = 0


# -----------------------------
# Training ensemble
# -----------------------------
print("\nRunning ElasticNet ensemble...")

for alpha in alphas:

    for seed in seeds:

        print(f"\nModel alpha={alpha} seed={seed}")

        fold_preds = np.zeros(len(X_test))
        fold_scores = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(X)):

            print(f"\nFold {fold+1}")

            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            model = ElasticNet(
                alpha=alpha,
                l1_ratio=0.95,
                max_iter=20000,
                tol=1e-3,
                selection="random",
                random_state=seed
            )

            model.fit(X_train, y_train)

            preds = model.predict(X_val)

            rmse = np.sqrt(mean_squared_error(y_val, preds))
            fold_scores.append(rmse)

            print("Fold RMSE:", rmse)

            # Diagnostic metrics
            metric_diagnostics(y_val, preds)

            fold_preds += model.predict(X_test) / kf.n_splits

        print("\nCV RMSE:", np.mean(fold_scores))

        test_preds += fold_preds
        total_models += 1


# -----------------------------
# Average ensemble predictions
# -----------------------------
test_preds /= total_models


# -----------------------------
# Save submission
# -----------------------------
os.makedirs("../submissions", exist_ok=True)

submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": test_preds
})

path = "../submissions/exp20_1_elasticnet_big_ensemble.csv"

submission.to_csv(path, index=False, header=False)

print("\nSaved submission:", path)
print(pd.read_csv(path, header=None).head())

Loading data...
Train shape: (1322, 1559)
Test shape: (550, 1558)
Spectral features: 1555
Scaling features...

Running ElasticNet ensemble...

Model alpha=0.001 seed=42

Fold 1
Fold RMSE: 12.639114046081191

Metric diagnostics
------------------
RMSE: 12.639114046081191
MAE: 8.279134352134719
RMSLE: 0.3442581233780639
NRMSE (mean): 0.23988782936802694
NRMSE (range): 0.04520579687759816

Fold 2
Fold RMSE: 12.99409023629002

Metric diagnostics
------------------
RMSE: 12.99409023629002
MAE: 8.112758913034112
RMSLE: 0.3419840999340805
NRMSE (mean): 0.25125916458030456
NRMSE (range): 0.04521382372008681

Fold 3
Fold RMSE: 11.550010756649385

Metric diagnostics
------------------
RMSE: 11.550010756649385
MAE: 7.471565064674492
RMSLE: 0.5080206381464254
NRMSE (mean): 0.23741385092632425
NRMSE (range): 0.04275611903867123

Fold 4
Fold RMSE: 12.915375018342797

Metric diagnostics
------------------
RMSE: 12.915375018342797
MAE: 8.50307285456843
RMSLE: 0.4577939742790837
NRMSE (mean): 0.2857565